In [ ]:
#In bronze we are reading in the json raw data and returning 2 parquet files playlist and info

In [13]:
import json
import pandas
from pathlib import Path
import os
import pyarrow
from pyarrow import parquet

In [10]:
#setting cwd to data_engineering_spotify_project so that notation is convenient as its the outermost file
os.chdir(r"C:\Users\Filipo\Desktop\data_engineering_spotify_project")

In [5]:
#setting up lists for both info and playlist
info_list = list()
playlist_list = list()

#iterating over all our data json slices in data folder
for x in Path('bronze/data').glob('*.json'):
    #converting x  from path to file object so the data in x gets stored as a python dictionary    with Path.open(x) as f_obj:
    with Path.open(x) as f_obj:
        data_py_v = json.load(f_obj)
        info_py_v = data_py_v['info']
        playlists_py_v = data_py_v['playlists']

        #adding the info data from slice to our info_list
        info_list.append(info_py_v)

        #our playlist info is in a list so we need to iterate over list to add each playlist individually instead of the entire list to our list
        for it_v in playlists_py_v:
            playlist_list.append(it_v)



In [ ]:
# print(info_list)

#first 2 elements of playlist_list
print(playlist_list[0:2])


In [7]:
#converting our info and playlist list to pandas DataFrames
#info list is a list of dictionaries
#playlist list is a list of dictionaries we will convert both of these to python tables
info_list_df = pandas.DataFrame(info_list)
playlist_list_df = pandas.DataFrame(playlist_list)

#checking out the dictionaries
# print(info_list_df.columns)
# print(playlist_list_df.columns)
# print(info_list_df.info())
# print(play_list_df.info())


#default is 5
# print(playlist_list_df.head(10))
# print(info_list_df.head(10))

#shape is a property of a dataframe not a function hence no parantheses. same as columns
# print(info_list.shape)
# print(playlist_list_df.shape)

#isnull() is a boolena materix across entire dataframe is record is null then true otherwise false
#sum of nulls across columns
# print(info_list_df.isnull().sum())
# print(playlist_list_df.isnull().sum())




In [14]:
#converting dataframes to parquet

#info_list_df.to_parquet('bronze/data/parquet_files/info.parquet')


#the playlist_list_df dataframe is too big to load it to a parquet file in one go
#playlist_list_df.to_parquet("bronze/data/parquet_files/playlists.parquet")


#acessing the first json slice to provide the parquet writer with the schema used in the json data
#this gives us the path of the first json slice
json_paths_v = Path('bronze/data').glob('*.json')
#this moves the current path iteration one ahead and stores the succesive path to first_slice_path_v
first_slice_path_v = next(json_paths_v)
with Path.open(first_slice_path_v) as file_v:
    first_slice_v = json.load(file_v)
    first_slice_playlists_v = first_slice_v['playlists']
    first_slice_arr_table_v = pyarrow.Table.from_pandas(pandas.DataFrame(first_slice_playlists_v))
    first_slice_pyarr_schema_v = first_slice_arr_table_v.schema


#now that we have the schema of the playlist portion of the json data we can use it in our parquet writer and write each playlist block of each slice into our playlist parquet file
with pyarrow.parquet.ParquetWriter('bronze/data/parquet_files/playlists.parquet' , first_slice_pyarr_schema_v) as pq_writer:
    #writing the arrow table based on the first json slice into our paruqet file
    pq_writer.write_table(first_slice_arr_table_v)
    for x in json_paths_v:
        #converting x  from path to file object so the data in x gets stored as a python dictionary with Path.open(x) as f_obj:
        with Path.open(x) as f_obj:
            data_py_v = json.load(f_obj)
            playlists_py_v = data_py_v['playlists']
            playlist_slice_df = pandas.DataFrame(playlists_py_v)
            #creating a arrow table based on current json slice
            pyarr_table_v = pyarrow.Table.from_pandas(playlist_slice_df)
            #writing arrow table to paruqet file with the use of a parquet writer
            pq_writer.write_table(pyarr_table_v)

#pq_writer objects gets terminated after scope
